# E0: SciBERT — обучение в Google Colab

Ноутбук самостоятельный: папка `src/`, ZIP с кодом и отдельный файл конфигурации не требуются.

**Запуск:** выбирается среда Colab с GPU, ячейки выполняются сверху вниз. В ячейке загрузки выбирается один архив `e0_scibert_data.zip` или одновременно два файла `train.jsonl` и `validation.jsonl`. В последней ячейке скачивается архив модели и результатов.

Дообучаются все веса SciBERT и выходной классификатор на 17 BIO-меток. Лучшая эпоха выбирается по микро-F1 сущностей на validation.

Используется только синтетический корпус. SciBERT ориентирован на английские научные тексты; его результат на русском учебном корпусе не подтверждает качество на реальных медицинских записях. Ноутбук подготовлен без локального запуска обучения.

Ноутбук использует Transformers 5.0.0 и PyTorch 2.6 или новее. Если Colab предложит перезапустить сеанс после установки, выполните ячейки повторно сверху вниз.


In [ ]:
%pip install -q "transformers==5.0.0" "torch>=2.6"


In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import hashlib
import json
import platform
import shutil
import tempfile
from pathlib import Path

import torch
import transformers
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForTokenClassification, AutoTokenizer,
    DataCollatorForTokenClassification, set_seed,
)
from google.colab import files

## 1. Параметры

Параметры дообучения совпадают с RuBERT. Модель и токенизатор SciBERT загружаются из фиксированной ревизии. При равном микро-F1 на validation выбирается более ранняя эпоха.

In [ ]:
config = {
    'model_id': 'allenai/scibert_scivocab_uncased',
    'revision': '24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1',
    'seed': 42,
    'epochs': 6,
    'batch_size': 4,
    'learning_rate': 2e-5,
    'weight_decay': 0.01,
    'max_length': 256,
    'max_grad_norm': 1.0,
    'num_threads': 4,
}

torch.set_num_threads(config["num_threads"])
torch.use_deterministic_algorithms(True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")

## 2. Загрузка корпуса

Загружается архив `e0_scibert_data.zip` с двумя JSONL-файлами либо два файла по отдельности. Архив читается в памяти без распаковки на диск. Каждая строка содержит `text`, `tokens`, `bio_tags`, `entities`, `id` и `template_id`. Проверяется отсутствие общих шаблонов и одинаковых текстов между train и validation.

In [ ]:
import io
import zipfile

uploaded = files.upload()
required = {"train.jsonl", "validation.jsonl"}
if len(uploaded) == 1 and next(iter(uploaded)).lower().endswith(".zip"):
    with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
        if len(archive.namelist()) != 2 or set(archive.namelist()) != required:
            raise ValueError("ZIP должен содержать только train.jsonl и validation.jsonl без подпапок")
        uploaded = {name: archive.read(name) for name in required}
if set(uploaded) != required:
    raise ValueError("Нужно загрузить e0_scibert_data.zip или два файла: train.jsonl и validation.jsonl")

corpus = {
    name: [json.loads(line) for line in uploaded[name].decode("utf-8").splitlines()]
    for name in required
}
train_records = corpus["train.jsonl"]
validation_records = corpus["validation.jsonl"]
if not train_records or not validation_records:
    raise ValueError("Обучающая и проверочная выборки должны быть непустыми")
for rows in (train_records, validation_records):
    for record in rows:
        if not isinstance(record.get("text"), str) or not record["text"].strip():
            raise ValueError("Запись должна содержать непустой текст")
        if len(record["tokens"]) != len(record["bio_tags"]):
            raise ValueError("Количество токенов и BIO-меток различается")
        for token in record["tokens"]:
            if not 0 <= token["start"] < token["end"] <= len(record["text"]):
                raise ValueError("Диапазон токена вне текста")
            if record["text"][token["start"]:token["end"]] != token["text"]:
                raise ValueError("Позиции токена не соответствуют тексту")
if {r["template_id"] for r in train_records} & {r["template_id"] for r in validation_records}:
    raise ValueError("Шаблоны train и validation пересекаются")
if {r["text"] for r in train_records} & {r["text"] for r in validation_records}:
    raise ValueError("Тексты train и validation пересекаются")
print("Train:", len(train_records), "Validation:", len(validation_records))

## 3. Подготовка меток и восстановление сущностей

Первый подтокен получает исходную BIO-метку, последующие подтокены сущности — I-метку. Для обычного текста сохраняется O. Специальные токены получают `-100`; дополнение с такой же меткой добавляется при сборке пакетов.

Позиции подтокенов сохраняются относительно исходного текста. Одиночная I-метка считается началом сущности. Длинные записи не обрезаются молча: превышение `max_length` вызывает ошибку.

In [ ]:
LABELS = ("NAME", "DATE", "PHONE", "EMAIL", "ID", "LOCATION", "ORGANIZATION", "AGE")

BIO_LABELS = ("O",) + tuple(prefix + label for label in LABELS for prefix in ("B-", "I-"))
LABEL2ID = {label: index for index, label in enumerate(BIO_LABELS)}
ID2LABEL = {index: label for label, index in LABEL2ID.items()}


def align_labels(word_labels, word_ids):
    """Последующие подтокены сущности получают I-метку, служебные — -100."""
    result = []
    previous = None
    for word_id in word_ids:
        if word_id is None:
            result.append(-100)
        else:
            if type(word_id) is not int or not 0 <= word_id < len(word_labels):
                raise ValueError("Подтокен ссылается на отсутствующее слово")
            label = word_labels[word_id]
            if word_id == previous and label.startswith("B-"):
                label = "I-" + label[2:]
            if label not in LABEL2ID:
                raise ValueError("Неизвестная BIO-метка: " + label)
            result.append(LABEL2ID[label])
        previous = word_id
    return result


def encode_record(record, tokenizer, max_length):
    """Подготовка входов и абсолютных позиций без скрытого обрезания текста."""
    if not tokenizer.is_fast:
        raise ValueError("Требуется быстрый токенизатор с word_ids()")
    words = [token["text"] for token in record["tokens"]]
    encoded = tokenizer(
        words, is_split_into_words=True, return_offsets_mapping=True,
        truncation=False, padding=False,
    )
    if len(encoded["input_ids"]) > max_length:
        raise ValueError("Запись {} превышает max_length={}".format(
            record.get("id", "без идентификатора"), max_length
        ))
    word_ids = encoded.word_ids()
    if {word_id for word_id in word_ids if word_id is not None} != set(range(len(words))):
        raise ValueError("Токенизатор пропустил исходные слова")
    offsets = encoded.pop("offset_mapping")
    spans = []
    for word_id, (start, end) in zip(word_ids, offsets):
        if word_id is None:
            spans.append(None)
            continue
        if not 0 <= start < end <= len(words[word_id]):
            raise ValueError("Некорректные позиции подтокена")
        word_start = record["tokens"][word_id]["start"]
        spans.append((word_start + start, word_start + end))
    features = dict(encoded)
    features["labels"] = align_labels(record["bio_tags"], word_ids)
    return features, spans


def decode_entities(predictions, spans):
    """Восстановление сущностей; одиночная I-метка считается началом сущности."""
    if len(predictions) != len(spans):
        raise ValueError("Число предсказаний и диапазонов различается")
    entities = []
    current = None
    previous_end = 0
    for prediction, span in zip(predictions, spans):
        if span is None:
            current = None
            continue
        if prediction not in ID2LABEL:
            raise ValueError("Неизвестный номер BIO-метки")
        start, end = span
        if start < previous_end or start >= end:
            raise ValueError("Диапазоны подтокенов пересекаются или пусты")
        previous_end = end
        tag = ID2LABEL[prediction]
        if tag == "O":
            current = None
            continue
        label = tag[2:]
        if tag.startswith("I-") and current is not None and current["label"] == label:
            current["end"] = end
        else:
            current = {"start": start, "end": end, "label": label}
            entities.append(current)
    return entities

## 4. Метрики

Сущность засчитывается только при совпадении типа, начала и конца. Вычисляются precision, recall, F1 и число пропусков по восьми классам, микро- и макроусреднение. При нулевом знаменателе значение метрики равно нулю. Доля правильно размеченных подтокенов используется только как дополнительный показатель.

In [ ]:
def entity_keys(entities, text):
    """Проверка диапазонов и преобразование сущностей для строгого сравнения."""
    if not isinstance(entities, list):
        raise ValueError("Сущности должны быть списком")
    keys = set()
    for entity in entities:
        if not isinstance(entity, dict):
            raise ValueError("Сущность должна быть объектом")
        start, end, label = (entity.get(key) for key in ("start", "end", "label"))
        if type(start) is not int or type(end) is not int:
            raise ValueError("Позиции сущности должны быть целыми числами")
        if not 0 <= start < end <= len(text) or label not in LABELS:
            raise ValueError("Недопустимый диапазон или класс сущности")
        key = (start, end, label)
        if key in keys:
            raise ValueError("Повтор сущности с теми же границами и типом")
        keys.add(key)
    return keys


def calculate_metrics(counts):
    """Расчёт метрик; при нулевом знаменателе результат принят равным нулю."""
    tp, fp, fn = (counts[key] for key in ("tp", "fp", "fn"))
    return dict(
        counts,
        precision=tp / (tp + fp) if tp + fp else 0.0,
        recall=tp / (tp + fn) if tp + fn else 0.0,
        f1=2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0,
    )



def evaluate_predictions(records, predictions):
    if len(records) != len(predictions):
        raise ValueError("Число предсказаний не совпадает с числом записей")
    counts = {label: {"tp": 0, "fp": 0, "fn": 0} for label in LABELS}
    for record, predicted in zip(records, predictions):
        expected_keys = entity_keys(record["entities"], record["text"])
        predicted_keys = entity_keys(predicted, record["text"])
        for key, entities in (
            ("tp", expected_keys & predicted_keys),
            ("fp", predicted_keys - expected_keys),
            ("fn", expected_keys - predicted_keys),
        ):
            for start, end, label in entities:
                counts[label][key] += 1
    per_class = {label: calculate_metrics(counts[label]) for label in LABELS}
    micro = calculate_metrics({
        key: sum(counts[label][key] for label in LABELS) for key in ("tp", "fp", "fn")
    })
    macro = {
        key: sum(per_class[label][key] for label in LABELS) / len(LABELS)
        for key in ("precision", "recall", "f1")
    }
    return {"records": len(records), "per_class": per_class, "micro": micro, "macro": macro}

## 5. Дообучение SciBERT

Пакеты дополняются до общей длины. При оценке позиции `-100` пропускаются, а сущности сравниваются по типу и символьным границам. После каждой эпохи вычисляется микро-F1 на validation. Контрольная точка сохраняется при улучшении этой метрики и содержит веса всей модели.

In [ ]:
def make_loader(features, tokenizer, config, shuffle=False):
    return DataLoader(
        features,
        batch_size=config["batch_size"],
        shuffle=shuffle,
        generator=torch.Generator().manual_seed(config["seed"]),
        collate_fn=DataCollatorForTokenClassification(
            tokenizer=tokenizer, label_pad_token_id=-100,
        ),
    )


def evaluate_model(model, loader, records, spans, device):
    model.eval()
    predictions = []
    correct = total = 0
    with torch.inference_mode():
        for batch in loader:
            targets = batch["labels"].to(device)
            inputs = {key: value.to(device) for key, value in batch.items() if key != "labels"}
            predicted = model(**inputs).logits.argmax(dim=-1)
            active = targets != -100
            correct += ((predicted == targets) & active).sum().item()
            total += active.sum().item()
            for row in predicted.cpu().tolist():
                offsets = spans[len(predictions)]
                predictions.append(decode_entities(row[:len(offsets)], offsets))
    metrics = evaluate_predictions(records, predictions)
    metrics["token_accuracy"] = correct / total if total else 0.0
    return metrics, predictions


def train_model(model, tokenizer, train_records, validation_records, config, device, checkpoint):
    if config["max_length"] > model.config.max_position_embeddings:
        raise ValueError("max_length превышает контекстное окно модели")
    train_features = [encode_record(r, tokenizer, config["max_length"])[0] for r in train_records]
    validation_encoded = [encode_record(r, tokenizer, config["max_length"]) for r in validation_records]
    validation_features = [item[0] for item in validation_encoded]
    validation_spans = [item[1] for item in validation_encoded]
    train_loader = make_loader(train_features, tokenizer, config, shuffle=True)
    validation_loader = make_loader(validation_features, tokenizer, config)

    model.to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )
    best_score = -1.0
    best_metrics = best_predictions = None
    best_epoch = None
    history = []
    for epoch in range(1, config["epochs"] + 1):
        model.train()
        loss_sum = token_count = 0
        for batch in train_loader:
            batch = {key: value.to(device) for key, value in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            loss = model(**batch).loss
            if not torch.isfinite(loss):
                raise ValueError("Получена неконечная функция потерь")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["max_grad_norm"])
            optimizer.step()
            active_count = (batch["labels"] != -100).sum().item()
            loss_sum += loss.item() * active_count
            token_count += active_count
        metrics, predictions = evaluate_model(
            model, validation_loader, validation_records, validation_spans, device,
        )
        score = metrics["micro"]["f1"]
        history.append({
            "epoch": epoch, "train_loss": loss_sum / token_count,
            "validation_micro_f1": score,
        })
        print("Эпоха {}: loss={:.4f}, validation F1={:.4f}".format(
            epoch, history[-1]["train_loss"], score,
        ))
        if score > best_score:
            best_score = score
            model.save_pretrained(checkpoint)
            best_epoch = epoch
            best_metrics = metrics
            best_predictions = predictions
    tokenizer.save_pretrained(checkpoint)
    return best_metrics, best_predictions, history, best_epoch

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    config["model_id"], revision=config["revision"], use_fast=True, do_lower_case=True,
)

In [ ]:
set_seed(config["seed"])
output_dir = Path(tempfile.mkdtemp(prefix="e0_scibert_", dir="/content"))
checkpoint = output_dir / "checkpoint"
model = AutoModelForTokenClassification.from_pretrained(
    config["model_id"], revision=config["revision"],
    num_labels=len(BIO_LABELS), id2label=ID2LABEL, label2id=LABEL2ID,
    attn_implementation="eager",
)
metrics, predictions, history, best_epoch = train_model(
    model, tokenizer, train_records, validation_records, config, device, checkpoint,
)
del model
print("Выбранная эпоха:", best_epoch)
print("Класс          Precision  Recall     F1         FN")
for label, values in metrics["per_class"].items():
    print("{:<14} {:<10.4f} {:<10.4f} {:<10.4f} {}".format(
        label, values["precision"], values["recall"], values["f1"], values["fn"],
    ))
for average in ("micro", "macro"):
    print(average, metrics[average])
print("Доля правильно размеченных подтокенов:", metrics["token_accuracy"])

## 6. Сохранение результата

Скачивается один архив: лучшая модель, токенизатор, параметры, версии библиотек, описание оборудования, хеши двух входных файлов, журнал эпох, метрики и предсказания validation. Полноразмерные веса не включаются в Git.

In [ ]:
environment = {
    "python": platform.python_version(), "platform": platform.platform(),
    "torch": torch.__version__, "transformers": transformers.__version__,
    "tokenizers": __import__("tokenizers").__version__,
    "device": str(device),
    "device_name": torch.cuda.get_device_name(0) if device.type == "cuda" else platform.processor(),
    "cpu_threads": torch.get_num_threads(), "cuda": torch.version.cuda,
}
input_hashes = {
    name: hashlib.sha256(content).hexdigest() for name, content in uploaded.items()
}
result = {
    "experiment": "E0-SciBERT", "split": "validation",
    "training_mode": "full_fine_tuning", "selection_metric": "validation.micro.f1",
    "config": config, "label2id": LABEL2ID, "environment": environment,
    "input_sha256": input_hashes, "best_epoch": best_epoch,
    "history": history, "metrics": metrics,
}
(output_dir / "e0-scibert-validation.json").write_text(
    json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8",
)
(output_dir / "validation-predictions.jsonl").write_text(
    "".join(json.dumps({"id": record["id"], "entities": predicted}, ensure_ascii=False) + "\n"
            for record, predicted in zip(validation_records, predictions)), encoding="utf-8",
)
archive_path = shutil.make_archive("/content/e0_scibert_results", "zip", output_dir)
files.download(archive_path)

## Ограничения

SciBERT предобучен на английских научных текстах. Русскоязычный синтетический корпус небольшой и шаблонный. Записи длиннее `max_length` отклоняются. В архиве сохраняются метрики на validation.

Pytest-тесты хранятся отдельно в папке `test/` проекта и не запускаются при выполнении ноутбука.

Источники: [SciBERT](https://huggingface.co/allenai/scibert_scivocab_uncased), [классификация токенов](https://huggingface.co/docs/transformers/tasks/token_classification).